In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('250122_인천시데이터.csv')
# confirm_date를 datetime 형식으로 변환
df['confirm_date'] = pd.to_datetime(df['confirm_date'])

# Step 1: 기존 필터링 방법 적용
filtered_cluster_ids = []
for cluster_id in df['transmission_cluster'].unique():
    cluster = df[df['transmission_cluster'] == cluster_id]
    min_date = cluster['confirm_date'].min()
    four_days_later = min_date + pd.Timedelta(days=4)

    # 첫날 이후 4일 이내에 새로운 데이터가 있는지 확인
    mask = (cluster['confirm_date'] > min_date) & (cluster['confirm_date'] <= four_days_later)

    # 4일 내 데이터가 있거나, 클러스터에 데이터가 첫날만 있는 경우 유지
    if cluster[mask].shape[0] > 0 or cluster.shape[0] == 1:
        filtered_cluster_ids.append(cluster_id)

# 필터링된 데이터셋 생성
filtered_data = df[df['transmission_cluster'].isin(filtered_cluster_ids)].copy()

# Step 2: 클러스터 크기가 192 이상인 것 제거
cluster_sizes = filtered_data.groupby('transmission_cluster').size().reset_index(name='row_count')
clusters_to_remove = cluster_sizes[cluster_sizes['row_count'] >= 101]['transmission_cluster'].unique()

filtered_data = filtered_data[~filtered_data['transmission_cluster'].isin(clusters_to_remove)]

# Step 3: y, d 할당
all_data = []

for orig_cluster_id in sorted(filtered_data['transmission_cluster'].unique()):
    cluster = filtered_data[filtered_data['transmission_cluster'] == orig_cluster_id].copy()
    cluster = cluster.sort_values('confirm_date').reset_index(drop=True)

    min_date = cluster['confirm_date'].min()
    max_date = cluster['confirm_date'].max()

    cluster['diff_days'] = (cluster['confirm_date'] - min_date).dt.days 
    cluster['gender'] = cluster['gender'].map({'남': 1, '여': 0})

    cut_step = 1
    current_cut_date = min_date

    while cut_step <= 5 and current_cut_date <= max_date:
        subset = cluster[cluster['confirm_date'] <= current_cut_date].copy()
        if subset.empty:
            break

        future_cases = cluster[cluster['confirm_date'] > current_cut_date]
        y_count = len(future_cases)
        d_value = (future_cases['confirm_date'].max() - current_cut_date).days if not future_cases.empty else 0

        subset['cut_date'] = cut_step
        subset['y'] = y_count
        subset['d'] = d_value

        all_data.append(subset)

        current_cut_date += pd.Timedelta(days=1)
        cut_step += 1

        if y_count == 0 and d_value == 0:
            break

final_df = pd.concat(all_data, ignore_index=True).drop('confirm_date', axis=1)

# Step 4: transmission_cluster 재할당 (Re-indexing)
unique_clusters = final_df['transmission_cluster'].unique()
cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(unique_clusters), start=1)}
final_df = final_df.sort_values(['transmission_cluster', 'cut_date'])
final_df['transmission_cluster'] = final_df.groupby(['transmission_cluster', 'cut_date']).ngroup()

# 최종 결과 확인
print(final_df.head(10))




   gender  test_R  test_E  symptom  distance    danger  \
0       1   20.00   21.60        1         1  5.530419   
1       0   26.00   28.00        0         3  5.530419   
2       0   18.90   20.70        1         3  5.030419   
3       1   16.00   17.80        1         3  5.553146   
4       1   12.20   14.20        1         3  5.030419   
5       1   24.60   24.80        0         3  5.338111   
6       0   18.50   19.70        1         3  5.553146   
7       1   28.86   26.81        0         4  6.137561   
8       1   28.38   25.65        1         4  7.101847   
9       0   17.60   19.40        1         4  6.137561   

   birthyear_category_1930-1959  birthyear_category_1960-1989  \
0                             1                             0   
1                             1                             0   
2                             1                             0   
3                             0                             1   
4                             0     

In [ ]:
import numpy as np
# ----------------------------------------------
# 추가: danger_prime 생성 및 상관관계 출력
# ----------------------------------------------

# [가정] 원본 데이터(df) 혹은 전처리된 데이터(final_df)에 'danger'와 'distance' 컬럼이 존재한다고 가정합니다.
# 예시로, danger_prime을 np.log1p 변환을 통해 생성합니다.
# (만약 다른 변환 방식이 필요하면 아래 코드를 수정하세요.)

final_df['danger_prime'] = np.log1p(final_df['danger'])

# danger_prime과 y, d, distance 간의 상관계수 계산
corr_danger_y = final_df['danger_prime'].corr(final_df['y'])
corr_danger_d = final_df['danger_prime'].corr(final_df['d'])
corr_danger_distance = final_df['danger_prime'].corr(final_df['distance'])

print("\n상관관계 결과:")
print("danger_prime과 y의 상관관계:", corr_danger_y)
print("danger_prime과 d의 상관관계:", corr_danger_d)
print("danger_prime과 distance의 상관관계:", corr_danger_distance)

In [3]:
final_df.to_csv('preprocessed_incheon.csv', index=False)

In [12]:
final_df.transmission_cluster.unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [2]:
max_cluster_size = final_df.groupby('transmission_cluster').size().max()
print(max_cluster_size)

98


In [2]:
# Step 1: Filter clusters with no new cases within 4 days of the earliest date, but keep clusters with only the first day
filtered_cluster_ids = []
for cluster_id in data['transmission_cluster'].unique():
    cluster = data[data['transmission_cluster'] == cluster_id]
    min_date = cluster['confirm_date'].min()
    four_days_later = min_date + pd.Timedelta(days=4)
    
    # Check if there are any cases after the first day and within the next 4 days
    mask = (cluster['confirm_date'] > min_date) & (cluster['confirm_date'] <= four_days_later)
    
    # If there are no cases in the 4-day window, but the cluster has only the first day, keep it
    if cluster[mask].shape[0] > 0 or cluster.shape[0] == 1:
        filtered_cluster_ids.append(cluster_id)

filtered_data = data[data['transmission_cluster'].isin(filtered_cluster_ids)].copy()

In [3]:
filtered_data.to_csv('note_filtered_clusters.csv', index=False)

In [4]:

# 2. 클러스터 재인덱싱
original_clusters = sorted(filtered_data['transmission_cluster'].unique())
cluster_id_map = {old: new+1 for new, old in enumerate(original_clusters)}
filtered_data['original_cluster'] = filtered_data['transmission_cluster'].map(cluster_id_map)

In [15]:
all_data = []
# new_cluster_id는 기존 클러스터 id와 혼동되지 않도록 큰 값에서 시작
new_cluster_id = max(cluster_id_map.values()) * 100  
df = filtered_data
for orig_cluster_id in sorted(df['original_cluster'].unique()):
    # 해당 클러스터만 추출, 날짜순 정렬
    cluster = df[df['original_cluster'] == orig_cluster_id].copy()
    cluster = cluster.sort_values('confirm_date').reset_index(drop=True)
    
    # 최소/최대 날짜
    min_date = cluster['confirm_date'].min()
    max_date = cluster['confirm_date'].max()
    
    # diff_days를 1부터 시작하도록 (day1 -> 1, day3 -> 3, ...)
    cluster['diff_days'] = (cluster['confirm_date'] - min_date).dt.days + 1
    
    # 성별을 0/1로 변환
    cluster['gender'] = cluster['gender'].map({'남': 1, '여': 0})
    
    cut_step = 1
    current_cut_date = min_date
    
    # cut_date를 하루씩 증가시키며 최대 5번 반복
    while cut_step <= 5 and current_cut_date <= max_date:
        # 현재 cut_date 이하인 row들을 뽑기
        subset = cluster[cluster['confirm_date'] <= current_cut_date].copy()
        if subset.empty:
            break
        
        # cut_date 이후(즉 미래)에 해당하는 row들
        future_cases = cluster[cluster['confirm_date'] > current_cut_date]
        
        # y: 미래에 존재하는 row 수
        y_count = len(future_cases)
        # d: 미래 row가 있다면 (max_date - current_cut_date).days, 없으면 0
        d_value = (max_date - current_cut_date).days if y_count > 0 else 0
        
        # 새로운 컬럼들 할당
        subset['transmission_cluster'] = new_cluster_id
        subset['cut_date'] = cut_step
        subset['y'] = y_count
        subset['d'] = d_value
        
        # 결과 저장
        all_data.append(subset)
        
        # 다음 cut_date를 하루 증가, cut_step 증가, 새 클러스터 ID 도 1 증가
        current_cut_date += pd.Timedelta(days=1)
        cut_step += 1
        new_cluster_id += 1
        
        # 미래가 더 이상 없으면 중단
        if y_count == 0 and d_value == 0:
            break

final_df = pd.concat(all_data, ignore_index=True)

In [30]:
final_df.to_csv('note_preprocessed_clusters.csv', index=False)

In [16]:
max(final_df['y'])

717

In [11]:
final_df['transmission_cluster'].unique()

array([23100, 23101, 23102, ..., 24135, 24136, 24137], dtype=object)